# EXACT 2026 — Replay 100 câu BTC trên Google Colab T4

Notebook này chạy từ Drive trắng đến kết quả cuối cùng:

1. kiểm tra GPU T4;
2. clone GitHub và checkout runner đã kiểm định;
3. mount Drive, upload và xác minh hai log BTC;
4. cài dependency, tải Qwen 7B và semantic encoder đã pin;
5. chạy self-test, dry-run và smoke trên **public data**;
6. replay toàn bộ 100 câu BTC qua 7 ablation và thu telemetry.

> Protocol chính xác: **50 Type 1 × 3 variant + 50 Type 2 × 4 variant = 350 jobs**. Mỗi variant chạy một lần với greedy decoding (`temperature=0`). Không chạy lại latency riêng vì hai log đã có latency end-to-end chính thức.

> Đây là **post-hoc organizer-test replay**, không phải điểm official mới hay blind unseen-test. Chỉ các bảng aggregate được phép chia sẻ; raw audit files có thể chứa dữ liệu hidden.

In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
import torch

assert torch.cuda.is_available(), "Hãy chọn Runtime > Change runtime type > T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME} ({GPU_GIB:.2f} GiB)")
assert "T4" in GPU_NAME.upper(), f"Protocol yêu cầu T4, hiện tại là {GPU_NAME}"


## 1. Clone GitHub và pin runner

Notebook checkout commit runner đã qua self-test/dry-run. Không đổi commit giữa các phiên resume.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/EXACT2026-NeuroSymbolic-QA.git"
REPO_DIR = Path("/content/EXACT2026-NeuroSymbolic-QA")
BRANCH = "main"
VALIDATED_RUNNER_COMMIT = "031298bc63e3a3d48c24b10a1de53ed90becd9f7"
HIDDEN_LOG_NAMES = (
    "exact_eval_round1_Cay_Nha_La_Vuon.json",
    "exact_eval_round2_Cay_Nha_La_Vuon.json",
)

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)

exclude_path = REPO_DIR / ".git" / "info" / "exclude"
existing = exclude_path.read_text(encoding="utf-8") if exclude_path.exists() else ""
for name in HIDDEN_LOG_NAMES:
    if name not in existing.splitlines():
        existing += ("" if existing.endswith("\n") or not existing else "\n") + name + "\n"
exclude_path.write_text(existing, encoding="utf-8")

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", VALIDATED_RUNNER_COMMIT], check=True)
os.chdir(REPO_DIR)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert CODE_COMMIT == VALIDATED_RUNNER_COMMIT
print(f"Code commit: {CODE_COMMIT}")
subprocess.run(["git", "status", "--short"], check=True)


## 2. Mount Drive, cấu hình cache và upload hai log BTC

Hai JSON được giữ trong `MyDrive/EXACT2026-paper-inputs/`. Không commit, không public thư mục này. Cache model mặc định ở `/content`; bật cache Drive chỉ khi Drive còn hơn 20 GiB.

In [ ]:
from google.colab import drive, files
import hashlib, json, os, shutil
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
MY_DRIVE = Path("/content/drive/MyDrive")
INPUT_DIR = MY_DRIVE / "EXACT2026-paper-inputs"
RESULTS_ROOT = MY_DRIVE / "EXACT2026-paper-results"
PERSIST_MODEL_CACHE_ON_DRIVE = False
HF_CACHE_ROOT = MY_DRIVE / "EXACT2026-huggingface-cache" if PERSIST_MODEL_CACHE_ON_DRIVE else Path("/content/EXACT2026-huggingface-cache")
HF_HUB_CACHE = HF_CACHE_ROOT / "hub"
for directory in (INPUT_DIR, RESULTS_ROOT, HF_CACHE_ROOT, HF_HUB_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

PINNED_QWEN_COMMIT = "a09a35458c702b33eeacc393d103063234e8bc28"
PINNED_EMBEDDING_COMMIT = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
qwen_snapshot = HF_HUB_CACHE / "models--Qwen--Qwen2.5-7B-Instruct" / "snapshots" / PINNED_QWEN_COMMIT
encoder_snapshot = HF_HUB_CACHE / "models--sentence-transformers--all-MiniLM-L6-v2" / "snapshots" / PINNED_EMBEDDING_COMMIT
cache_ready = all(path.exists() for path in (
    qwen_snapshot / "model.safetensors.index.json",
    qwen_snapshot / "model-00001-of-00004.safetensors",
    qwen_snapshot / "model-00002-of-00004.safetensors",
    qwen_snapshot / "model-00003-of-00004.safetensors",
    qwen_snapshot / "model-00004-of-00004.safetensors",
    encoder_snapshot / "model.safetensors",
))
free_gib = shutil.disk_usage(HF_CACHE_ROOT).free / 1024**3
if not cache_ready and free_gib < 20:
    raise RuntimeError(f"Cache chỉ còn {free_gib:.1f} GiB; cần ít nhất 20 GiB")

os.environ["PAPER_OUTPUT_DIR"] = str(RESULTS_ROOT)
os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ROUND1_NAME = HIDDEN_LOG_NAMES[0]
ROUND2_NAME = HIDDEN_LOG_NAMES[1]
EXPECTED_LOG_IDENTITY = {
    ROUND1_NAME: {"sha256": "6d5e7a86a5e0a7ed1e1c3e9f43b7228bd930d0e4a7a6133f62ad302483b7fd4b", "round": "round1", "sample": "eval-round1-type1-type2-v2", "score": 39.38},
    ROUND2_NAME: {"sha256": "03032ec92f384d3c0ccf76e9f7801cf0b107452805b97115b00061e9c6fdc813", "round": "round2", "sample": "eval-round2-type1-type2-v3", "score": 44.8},
}
missing = [name for name in HIDDEN_LOG_NAMES if not (INPUT_DIR / name).exists()]
if missing:
    print("Upload đúng các file:", missing)
    uploaded = files.upload()
    for name in missing:
        if name not in uploaded:
            raise FileNotFoundError(f"Thiếu file {name}")
        (INPUT_DIR / name).write_bytes(uploaded[name])

ROUND1_LOG, ROUND2_LOG = INPUT_DIR / ROUND1_NAME, INPUT_DIR / ROUND2_NAME
for path in (ROUND1_LOG, ROUND2_LOG):
    payload = json.loads(path.read_text(encoding="utf-8"))
    expected = EXPECTED_LOG_IDENTITY[path.name]
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == expected["sha256"], f"Sai SHA-256: {path.name}"
    assert payload.get("eval_round") == expected["round"]
    assert payload.get("sample_version") == expected["sample"]
    assert len(payload.get("logs") or []) == 50
    assert sum(row.get("type") == "type1" for row in payload["logs"]) == 25
    assert sum(row.get("type") == "type2" for row in payload["logs"]) == 25
    assert payload.get("summary", {}).get("team") == "Cây Nhà Lá Vườn"
    assert float(payload["summary"].get("score", payload["summary"].get("total_points"))) == expected["score"]
    print(f"OK {path.name}: 50 records, sha256={digest[:16]}...")
    target = REPO_DIR / path.name
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        if hashlib.sha256(target.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Repo root đã có file khác nội dung: {target}")
        continue
    target.symlink_to(path)

print(f"Private inputs: {INPUT_DIR}")
print(f"Private results: {RESULTS_ROOT}")
print(f"HF cache: {HF_CACHE_ROOT} ({free_gib:.1f} GiB free)")


## 3. Cài dependency và khóa environment


In [ ]:
import hashlib, importlib.metadata, json, subprocess, sys

PACKAGE_SPECS = {
    "fastapi": "fastapi>=0.110", "pydantic": "pydantic>=2", "httpx": "httpx>=0.27",
    "pyyaml": "pyyaml>=6", "loguru": "loguru>=0.7", "openai": "openai>=1.30",
    "z3-solver": "z3-solver>=4.13", "sympy": "sympy>=1.12", "numpy": "numpy>=1.24,<3",
    "matplotlib": "matplotlib>=3.7", "transformers": "transformers>=4.46,<5",
    "accelerate": "accelerate>=0.30,<2", "bitsandbytes": "bitsandbytes>=0.43,<1",
    "faiss-cpu": "faiss-cpu>=1.8", "sentence-transformers": "sentence-transformers>=2.7,<6",
}
LOCKED_NAMES = tuple(dict.fromkeys((*PACKAGE_SPECS, "huggingface-hub", "tokenizers", "safetensors", "scipy", "scikit-learn")))
DEPENDENCY_LOCK = INPUT_DIR / "btc-replay-dependencies.lock.json"
if DEPENDENCY_LOCK.exists():
    locked = json.loads(DEPENDENCY_LOCK.read_text(encoding="utf-8"))
    missing_lock = sorted(set(LOCKED_NAMES) - set(locked))
    if missing_lock: raise RuntimeError(f"Dependency lock thiếu: {missing_lock}")
    install_specs = [f"{name}=={locked[name]}" for name in LOCKED_NAMES]
else:
    install_specs = list(PACKAGE_SPECS.values())
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *install_specs], check=True)

resolved_versions = {name: importlib.metadata.version(name) for name in LOCKED_NAMES}
if DEPENDENCY_LOCK.exists():
    mismatch = {name: (locked[name], version) for name, version in resolved_versions.items() if locked[name] != version}
    if mismatch: raise RuntimeError(f"Version khác dependency lock: {mismatch}")
else:
    DEPENDENCY_LOCK.write_text(json.dumps(resolved_versions, indent=2, sort_keys=True) + "\n", encoding="utf-8")

pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(pip_check.stdout.strip())
PIP_CHECK_LOG = INPUT_DIR / "btc-replay-pip-check.txt"
PIP_CHECK_LOG.write_text(pip_check.stdout, encoding="utf-8")
runtime = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": GPU_NAME, "packages": resolved_versions, "pip_check": pip_check.returncode}
ENV_FINGERPRINT = hashlib.sha256(json.dumps(runtime, sort_keys=True).encode()).hexdigest()[:10]
RUNTIME_MANIFEST = INPUT_DIR / f"btc-replay-runtime-{ENV_FINGERPRINT}.json"
RUNTIME_MANIFEST.write_text(json.dumps(runtime, indent=2) + "\n", encoding="utf-8")
print(f"Environment fingerprint: {ENV_FINGERPRINT}")


## 4. Tải model/encoder đã pin và xác minh FAISS

Qwen tải khoảng 15,2 GB checkpoint gốc rồi được load 4-bit NF4 khi inference.

In [ ]:
import gc, json, os, pickle
from pathlib import Path
import faiss, numpy as np
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_COMMIT = PINNED_QWEN_COMMIT
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL_COMMIT = PINNED_EMBEDDING_COMMIT
MODEL_SNAPSHOT = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_COMMIT, cache_dir=str(HF_HUB_CACHE), ignore_patterns=["*.bin", "*.msgpack", "*.h5", "*.ot", "original/*"]))
EMBEDDING_SNAPSHOT = Path(snapshot_download(repo_id=EMBEDDING_MODEL_ID, revision=EMBEDDING_MODEL_COMMIT, cache_dir=str(HF_HUB_CACHE), ignore_patterns=["onnx/*", "openvino/*", "*.bin", "*.h5", "*.ot", "*.msgpack"]))
print(f"Qwen snapshot: {MODEL_SNAPSHOT}")
print(f"Encoder snapshot: {EMBEDDING_SNAPSHOT}")

index_dir = REPO_DIR / "data/formula_index"
manifest = json.loads((index_dir / "encoder.json").read_text(encoding="utf-8"))
assert manifest["model"] == EMBEDDING_MODEL_ID and manifest["revision"] == EMBEDDING_MODEL_COMMIT
index = faiss.read_index(str(index_dir / "index.faiss"))
with (index_dir / "metadata.pkl").open("rb") as handle: docs = pickle.load(handle)
texts = [f"{doc['domain']}: {doc['formula_natural']} — {' '.join(doc.get('keywords', []))}" for doc in docs]
encoder = SentenceTransformer(EMBEDDING_MODEL_ID, revision=EMBEDDING_MODEL_COMMIT, cache_folder=str(HF_HUB_CACHE), device="cpu")
fresh = encoder.encode(texts, show_progress_bar=False).astype("float32")
stored = np.vstack([index.reconstruct(i) for i in range(index.ntotal)]).astype("float32")
np.testing.assert_allclose(stored, fresh, rtol=1e-5, atol=1e-5)
print(f"FAISS provenance PASS: {index.ntotal} vectors, max delta={np.max(np.abs(stored-fresh)):.3e}")
del encoder, fresh, stored
gc.collect()


## 5. Self-test và dry-run (không inference)


In [ ]:
import json, subprocess, sys

VALIDATION_DIR = RESULTS_ROOT / "notebook_validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
self_test = subprocess.run([sys.executable, "paper/run_paper_experiments.py", "--mode", "self-test", "--install-deps", "no"], cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(VALIDATION_DIR / f"self_test_{CODE_COMMIT[:8]}_{ENV_FINGERPRINT}.log").write_text(self_test.stdout, encoding="utf-8")
print(self_test.stdout)
self_test.check_returncode()

DRY_NAME = "btc100_dry_run"
dry_cmd = [sys.executable, "paper/run_paper_experiments.py", "--mode", "dry-run", "--evaluation-data", "btc-rounds", "--temperature", "0", "--repeats", "1", "--deterministic-repeats", "1", "--latency-samples", "0", "--round1-log", str(ROUND1_LOG), "--round2-log", str(ROUND2_LOG), "--output-dir", str(RESULTS_ROOT), "--run-name", DRY_NAME, "--install-deps", "no"]
subprocess.run(dry_cmd, cwd=REPO_DIR, check=True)
print("Dry-run PASS: loader/scorer/manifest dùng đúng 100 câu BTC, không gọi model.")


## 6. Smoke test Qwen trên public data

Smoke không dùng 100 câu test, tránh xem kết quả test trước full replay. Sau lần đầu thành công có thể đặt `RUN_PUBLIC_SMOKE=False`.

In [ ]:
RUN_PUBLIC_SMOKE = True
SMOKE_NAME = f"public_smoke_{MODEL_COMMIT[:8]}_{EMBEDDING_MODEL_COMMIT[:8]}_{ENV_FINGERPRINT}"
smoke_cmd = [sys.executable, "paper/run_paper_experiments.py", "--mode", "smoke", "--evaluation-data", "public", "--backend", "transformers", "--model", MODEL_ID, "--model-revision", MODEL_COMMIT, "--embedding-model", EMBEDDING_MODEL_ID, "--embedding-model-revision", EMBEDDING_MODEL_COMMIT, "--quantization", "4bit", "--temperature", "0", "--latency-samples", "2", "--round1-log", str(ROUND1_LOG), "--round2-log", str(ROUND2_LOG), "--output-dir", str(RESULTS_ROOT), "--run-name", SMOKE_NAME, "--install-deps", "no", "--progress-every", "1"]
if RUN_PUBLIC_SMOKE:
    subprocess.run(smoke_cmd, cwd=REPO_DIR, check=True, env=os.environ.copy())
    smoke_dirs = [p for p in RESULTS_ROOT.glob(SMOKE_NAME + "*") if (p / "metrics/quality_gate.json").exists()]
    smoke_dir = max(smoke_dirs, key=lambda p: (p / "metrics/quality_gate.json").stat().st_mtime)
    smoke_quality = json.loads((smoke_dir / "metrics/quality_gate.json").read_text(encoding="utf-8"))
    assert smoke_quality["complete_for_requested_subset"] and smoke_quality["expected_total"] == smoke_quality["completed_total"] == 60
    assert smoke_quality["failed_total"] == smoke_quality["infrastructure_failed_total"] == 0
    print("Public smoke PASS: 60/60")
else:
    print("Public smoke skipped.")


## 7. Full BTC replay — 350 jobs

Nếu Colab ngắt phiên, chạy lại các cell thiết lập, đặt `RUN_PUBLIC_SMOKE=False`, rồi chạy lại cell này. Giữ nguyên run name, commit, model và tham số để resume.

In [ ]:
assert CODE_COMMIT == VALIDATED_RUNNER_COMMIT
FULL_RUN_NAME = f"qwen25_7b_t4_btc100_once_{MODEL_COMMIT[:8]}_{EMBEDDING_MODEL_COMMIT[:8]}_{ENV_FINGERPRINT}"
full_cmd = [
    sys.executable, "paper/run_paper_experiments.py",
    "--mode", "full", "--evaluation-data", "btc-rounds", "--tracks", "both",
    "--backend", "transformers", "--model", MODEL_ID, "--model-revision", MODEL_COMMIT,
    "--embedding-model", EMBEDDING_MODEL_ID, "--embedding-model-revision", EMBEDDING_MODEL_COMMIT,
    "--quantization", "4bit", "--temperature", "0", "--max-tokens", "1024",
    "--llm-timeout", "120", "--code-timeout", "8", "--repeats", "1",
    "--deterministic-repeats", "1", "--seed", "2026", "--latency-samples", "0",
    "--bootstrap-samples", "2000", "--max-retries", "2", "--cache-shared-stages", "--resume",
    "--round1-log", str(ROUND1_LOG), "--round2-log", str(ROUND2_LOG),
    "--output-dir", str(RESULTS_ROOT), "--run-name", FULL_RUN_NAME,
    "--install-deps", "no", "--progress-every", "10",
]
print("Running 350-job BTC replay. Raw outputs remain private on Drive.")
result = subprocess.run(full_cmd, cwd=REPO_DIR, env=os.environ.copy())
if result.returncode == 0:
    print("BTC replay complete.")
elif result.returncode in (2, 130):
    print("Checkpoint đã lưu; chạy lại cùng cell để resume hoặc xem errors.jsonl.")
else:
    raise RuntimeError(f"Runner failed: exit {result.returncode}")


## 8. Xác minh quality gate và xem báo cáo aggregate


In [ ]:
import json, shutil
from IPython.display import Markdown, display

run_dirs = [p for p in RESULTS_ROOT.glob(FULL_RUN_NAME + "*") if (p / "run_config.json").exists()]
if not run_dirs: raise FileNotFoundError(FULL_RUN_NAME)
ACTIVE_RUN_DIR = max(run_dirs, key=lambda p: (p / "run_config.json").stat().st_mtime)
quality = json.loads((ACTIVE_RUN_DIR / "metrics/quality_gate.json").read_text(encoding="utf-8"))
config = json.loads((ACTIVE_RUN_DIR / "run_config.json").read_text(encoding="utf-8"))
environment = json.loads((ACTIVE_RUN_DIR / "environment.json").read_text(encoding="utf-8"))
args = config["arguments"]

expected_args = {"mode": "full", "evaluation_data": "btc-rounds", "tracks": "both", "backend": "transformers", "quantization": "4bit", "temperature": 0.0, "max_tokens": 1024, "repeats": 1, "deterministic_repeats": 1, "latency_samples": 0, "bootstrap_samples": 2000, "type1_limit": 0, "type2_limit": 0}
mismatch = {k: {"expected": v, "observed": args.get(k)} for k, v in expected_args.items() if args.get(k) != v}
assert not mismatch, mismatch
assert quality["paper_ready"] and quality["btc_replay_scope"]
assert quality["expected_total"] == quality["completed_total"] == 350
assert quality["failed_total"] == quality["infrastructure_failed_total"] == 0
assert len(quality["matrix"]) == 7 and all(row["expected"] == row["completed"] == 50 for row in quality["matrix"])
assert config["data_invariants"]["total"] == 100 and config["data_invariants"]["type1_total"] == config["data_invariants"]["type2_total"] == 50
assert config["data_invariants"]["round1_type1"] == config["data_invariants"]["round1_type2"] == 25
assert config["data_invariants"]["round2_type1"] == config["data_invariants"]["round2_type2"] == 25
assert config["resolved"]["resolved_model_revision"] == MODEL_COMMIT
assert config["resolved"]["resolved_embedding_model_revision"] == EMBEDDING_MODEL_COMMIT
assert environment["git"]["commit_sha"] == VALIDATED_RUNNER_COMMIT and environment["git"]["dirty"] is False
assert "T4" in environment["gpu"]["devices"][0]["name"].upper()
assert (ACTIVE_RUN_DIR / "PAPER_READY").exists() and (ACTIVE_RUN_DIR / "TEST_REPLAY_READY").exists()

repro = ACTIVE_RUN_DIR / "reproducibility"
repro.mkdir(exist_ok=True)
for source in (DEPENDENCY_LOCK, PIP_CHECK_LOG, RUNTIME_MANIFEST): shutil.copy2(source, repro / source.name)
print(f"REPLAY VERIFIED: {ACTIVE_RUN_DIR}")
display(Markdown((ACTIVE_RUN_DIR / "paper_results.md").read_text(encoding="utf-8")))


## Artifact và quyền riêng tư

Có thể dùng cho paper: `paper_results.md`, `metrics/*.csv|json`, `tables/*`, `figures/*` và privacy notice trong `cases/`.

**Không chia sẻ:** `predictions.jsonl`, `events.jsonl`, `errors.jsonl`, `stage_cache.jsonl`, hai log gốc hoặc toàn bộ Drive results folder. Chúng có thể chứa câu hỏi, premise, gold, generated code hoặc model output hidden.

Troubleshooting: T4 phải giữ `4bit`; khi ngắt phiên hãy resume với đúng config; sau smoke đầu tiên đặt `RUN_PUBLIC_SMOKE=False`; nếu semantic RAG không load, sửa dependency thay vì âm thầm đổi condition.

In [ ]:
log_path = ACTIVE_RUN_DIR / "logs/runner.log"
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-80:]))
